# Soft Actor Critic in Parallel

In [ ]:
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib
import random 
import time
#%matplotlib inline

from IPython.display import HTML

from pendulum_plant.pendulum_plant import PendulumPlant
from pendulum_plant.simulation import Simulator

from sac.sac_parallel_sequential import sac_trainer
from sac.sac_controller import SacController


Set parameters for pendulum and enviroment

In [ ]:
# pendulum parameters
mass = 0.06
length = 0.1
damping = 0.0004
torque_limit = 0.02
coulomb_fric = 0.0
inertia = mass*length**2
gravity = 9.81

# environment parameters
dt = 0.02
max_steps = 500 
reward_type = "combined_reward"
target = [np.pi, 0]
target_epsilon = [0.05, 0.05]
random_init = "everywhere"
integrator = "runge_kutta"


# Training

In [ ]:
# training parameters
log_dir = "log_data/sac_parallel_training"
base_log_dir = "log_data/sac_parallel_evaluation_gs_4000"
learning_rate = 0.0003
n_episodes=50
eval_every = 4
eval_episodes = 1
start_training = 2000 #changed from 2000
number_of_envs = 4
gradient_steps=1000 # changed from 1000
batch_size=256 # changed from 256
reward_limit=21_000
device = 'cuda'

ec_times = []
pu_times = []

def run_evaluation():
    for run_idx in range(5):
        print(f"Running evaluation run {run_idx}")
        log_dir = os.path.join(base_log_dir, f"run_{run_idx}")
        trainer = sac_trainer(log_dir=log_dir)
        trainer.init_pendulum(mass=mass,
                            length=length,
                            inertia=inertia,
                            damping=damping,
                            coulomb_friction=coulomb_fric,
                            gravity=gravity,
                            torque_limit=torque_limit)
    
        trainer.init_environment(dt=dt,
                                integrator=integrator,
                                max_steps=max_steps,
                                reward_type=reward_type,
                                target=target, 
                                state_target_epsilon=target_epsilon,
                                random_init=random_init,
                                state_representation=3,
                                n_envs=number_of_envs)
    
        trainer.init_agent(learning_rate=learning_rate,
                        warm_start=False,
                        warm_start_path="",
                        device=device,
                        verbose=1)
    
        print(f"Training using {device}")
        start_time = time.time()
        trainer.train(n_episodes=n_episodes,
                    gradient_steps=gradient_steps,
                    batch_size=batch_size,
                    eval_every=eval_every,
                    eval_episodes=eval_episodes,
                    start_training=start_training,
                    save_path=log_dir,
                    reward_limit=reward_limit)
        print(f"Training total time  with {device}: {time.time()-start_time}")



# ------ n env evaluation --------
base_log_dir = "log_data/sac_parallel_evaluation_n_env_3"
n_episodes=67
number_of_envs = 3 # changed from 4
run_evaluation()

base_log_dir = "log_data/sac_parallel_evaluation_n_env_2"
number_of_envs = 2 # changed from 4
n_episodes=100
run_evaluation()

base_log_dir = "log_data/sac_parallel_evaluation_n_env_1"
n_episodes=200
number_of_envs = 1 # changed from 4
run_evaluation()

base_log_dir = "log_data/sac_parallel_evaluation_n_env_6"
n_episodes=50
number_of_envs = 6 # changed from 4
run_evaluation()

number_of_envs = 4 # changed from 4 RESETED
n_episodes=50 # RESETTED

# ---- Batch size evaluation -----
batch_size=512
base_log_dir = "log_data/sac_parallel_evaluation_bs_512"
run_evaluation()

batch_size=1024
base_log_dir = "log_data/sac_parallel_evaluation_bs_1024"
run_evaluation()

batch_size=128
base_log_dir = "log_data/sac_parallel_evaluation_bs_128"
run_evaluation()

batch_size=1024
gradient_steps=250
base_log_dir = "log_data/sac_parallel_evaluation_bs_1024_gs_250"
run_evaluation()

batch_size = 256 # RESET
gradient_steps = 1000 # RESET

# ---- Last gs evaluation -------
base_log_dir = "log_data/sac_parallel_evaluation_gs_4000"
gradient_steps=4000 # changed from 1000
run_evaluation()

gradient_steps=1000 # changed from 1000 RESET

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

base_log_dir = "log_data/sac_parallel_evaluation_gs_500"
n_repeats = 5
number_of_envs = 4

# Prepare empty lists to hold dataframes from all runs
train_dfs = []
eval_dfs = []

for run_idx in range(n_repeats):
    log_dir = os.path.join(base_log_dir, f"run_{run_idx}")
    train_path = os.path.join(log_dir, "train_rewards.csv")
    eval_path = os.path.join(log_dir, "eval_rewards.csv")

    # Load CSVs, add run index column
    train_df = pd.read_csv(train_path)
    train_df['run'] = run_idx
    train_dfs.append(train_df)

    eval_df = pd.read_csv(eval_path)
    eval_df['run'] = run_idx
    eval_dfs.append(eval_df)

# Concatenate all runs into single dataframes
train_all = pd.concat(train_dfs, ignore_index=True)
eval_all = pd.concat(eval_dfs, ignore_index=True)

# Convert timestamp strings to datetime.time
train_all['timestamp'] = pd.to_datetime(train_all['timestamp'], format='%H:%M:%S')
eval_all['timestamp'] = pd.to_datetime(eval_all['timestamp'], format='%H:%M:%S')

# Compute elapsed seconds relative to first timestamp per run (to align time within each run)
def compute_elapsed(df):
    elapsed_list = []
    for run in df['run'].unique():
        subset = df[df['run'] == run]
        start = subset['timestamp'].min()
        elapsed = (subset['timestamp'] - start).dt.total_seconds()
        elapsed_list.append(elapsed)
    return pd.concat(elapsed_list).sort_index()

train_all['elapsed_seconds'] = compute_elapsed(train_all)
eval_all['elapsed_seconds'] = compute_elapsed(eval_all)

# ----- Plotting -----
plt.figure(figsize=(12, 18))

# 1) Training rewards
plt.subplot(4, 1, 1)
train_avg = train_all.groupby(['run', 'episode']).agg({
    'reward': 'mean',
    'elapsed_seconds': 'mean'
}).reset_index()
eval_avg = eval_all.groupby(['run', 'episode']).agg({
    'reward': 'mean',
    'elapsed_seconds': 'mean'
}).reset_index()

for run in train_avg['run'].unique():
    run_data = train_avg[train_avg['run'] == run]
    plt.plot(run_data['episode'], run_data['reward'], label=f'run {run}')

plt.title('Training reward averaged over 4 parallel episodes')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.legend(fontsize='small', ncol=2)

# 2) Training reward averaged over env_index and runs per episode vs elapsed time (mean)
plt.subplot(4, 1, 2)
train_avg = train_all.groupby(['run', 'episode']).agg({
    'reward': 'mean',
    'elapsed_seconds': 'mean'
}).reset_index()
for run in train_avg['run'].unique():
    run_data = train_avg[train_avg['run'] == run]
    plt.plot(run_data['elapsed_seconds'], run_data['reward'], label=f'run {run}')
plt.title('Training reward (avg over envs) vs elapsed time')
plt.xlabel('Elapsed time (seconds)')
plt.ylabel('Average reward')
plt.legend()

# 3) Eval reward averaged per episode and run vs elapsed time
plt.subplot(4, 1, 3)

for run in eval_avg['run'].unique():
    run_data = eval_avg[eval_avg['run'] == run]
    plt.plot(run_data['elapsed_seconds'], run_data['reward'], marker='o', label=f'run {run}')
plt.title('Evaluation reward (avg) vs elapsed time')
plt.xlabel('Elapsed time (seconds)')
plt.ylabel('Average eval reward')
plt.legend()

# 4) Eval reward averaged per run vs episode index
plt.subplot(4, 1, 4)
for run in eval_avg['run'].unique():
    run_data = eval_avg[eval_avg['run'] == run]
    plt.plot(run_data['episode'], run_data['reward'], marker='o', label=f'run {run}')
plt.title('Evaluation reward vs episode')
plt.xlabel('Episode')
plt.ylabel('Eval reward')
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import glob
from pathlib import Path

# -------- CONFIG --------
parent_dir = "log_data"  # Folder containing all experiment subfolders (_n_env_, _bs_, _gs_, etc.)
reward_limit = 21000
n_repeats = 5

# -------- FUNCTIONS --------
def load_runtime_log(exp_dir):
    """Load runtime_log.csv if exists"""
    path = os.path.join(exp_dir, "runtime_log.csv")
    if os.path.exists(path):
        df = pd.read_csv(path)
        df.columns = ["experience_collecting", "policy_updates", "evaluation"]  # rename to safe names
        df["total_time"] = df.sum(axis=1)
        return df
    return None

def rescale_episodes(df, exp_name):
    """
    Rescales episode numbers to match equivalent for n_envs = 4.
    Averages rewards over the grouping factor.
    """
    # Detect n_envs from folder name
    if "n_env_" in exp_name:
        try:
            n_envs = int(exp_name.split("n_env_")[1].split("_")[0])
        except ValueError:
            n_envs = 4
    else:
        n_envs = 4

    scale_factor = n_envs / 4.0  # how many envs relative to baseline
    if scale_factor != 1.0:
        # New "virtual episode" index
        df = df.copy()
        df["scaled_episode"] = (df["episode"] * scale_factor).astype(int)

        # Average rewards over these scaled episodes
        df = df.groupby(["run", "scaled_episode"], as_index=False)["reward"].mean()
        df.rename(columns={"scaled_episode": "episode"}, inplace=True)

    return df

def compute_episodes_to_target(train_df, reward_limit, exp_name):
    """Return number of episodes to reach target reward, rescaled to n_envs=4."""
    if "_n_env_" in exp_name:
        try:
            n_envs = int(exp_name.split("_n_env_")[1].split("_")[0])
        except ValueError:
            n_envs = 4
    else:
        n_envs = 4

    scale_factor = n_envs / 4.0

    episodes_needed = []
    for run in train_df["run"].unique():
        run_data = train_df[train_df["run"] == run]
        hit_idx = run_data[run_data["reward"] >= reward_limit]["episode"].min()
        if pd.isna(hit_idx):
            hit_idx = run_data["episode"].max()
        episodes_needed.append(hit_idx * scale_factor)

    return np.mean(episodes_needed), np.std(episodes_needed)

def load_train_eval(exp_dir, run_idx):
    """Load train_rewards.csv and eval_rewards.csv for one run"""
    train_path = os.path.join(exp_dir, f"run_{run_idx}", "train_rewards.csv")
    eval_path = os.path.join(exp_dir, f"run_{run_idx}", "eval_rewards.csv")

    if not os.path.exists(train_path) or not os.path.exists(eval_path):
        return None, None

    train_df = pd.read_csv(train_path)
    eval_df = pd.read_csv(eval_path)
    train_df["run"] = run_idx
    eval_df["run"] = run_idx
    return train_df, eval_df

def compute_target_proportion(eval_df):
    """Check proportion of runs hitting reward_limit early"""
    proportions = []
    for run in eval_df["run"].unique():
        run_data = eval_df[eval_df["run"] == run]
        if (run_data["reward"] >= reward_limit).any():
            proportions.append(1)
        else:
            proportions.append(0)
    return np.mean(proportions)

# -------- DATA AGGREGATION --------
all_results = []

# Iterate over experiment folders
for exp_folder in sorted(Path(parent_dir).glob("sac_parallel_evaluation_*")):
    exp_name = exp_folder.name
    exp_name = exp_name.split("sac_parallel_evaluation_")[1]
    exp_name = exp_name.replace("gs", "GradientSteps")
    exp_name = exp_name.replace("bs", "BatchSize")
    print(f"Processing {exp_name}")
    all_train, all_eval, all_runtime = [], [], []

    for run_idx in range(n_repeats):
        train_df, eval_df = load_train_eval(exp_folder, run_idx)
        if train_df is not None:
            train_df = rescale_episodes(train_df, exp_name)
            all_train.append(train_df)
        if eval_df is not None:
            eval_df = rescale_episodes(eval_df, exp_name)
            all_eval.append(eval_df)


    runtime_df = load_runtime_log(exp_folder)
    if runtime_df is not None:
        all_runtime.append(runtime_df)

    if not all_train or not all_eval:
        continue

    train_all = pd.concat(all_train, ignore_index=True)
    eval_all = pd.concat(all_eval, ignore_index=True)

    # Aggregate metrics
    train_rewards_mean = train_all.groupby("episode")["reward"].mean()
    train_rewards_std = train_all.groupby("episode")["reward"].std()

    eval_rewards_mean = eval_all.groupby("episode")["reward"].mean()
    eval_rewards_std = eval_all.groupby("episode")["reward"].std()

    target_prop = compute_target_proportion(eval_all)

    total_times = []
    if all_runtime:
        total_times = pd.concat(all_runtime)["total_time"].values
    
    ep_mean, ep_std = compute_episodes_to_target(train_all, reward_limit, exp_name)
    
    all_results.append({
        "name": exp_name,
        "train_mean": train_rewards_mean,
        "train_std": train_rewards_std,
        "eval_mean": eval_rewards_mean,
        "eval_std": eval_rewards_std,
        "target_prop": target_prop,
        "time_mean": np.mean(total_times) if len(total_times) > 0 else None,
        "time_std": np.std(total_times) if len(total_times) > 0 else None,
        "episodes_mean": ep_mean,
        "episodes_std": ep_std
    })

# -------- PLOTTING --------
fig, axes = plt.subplots(4, 1, figsize=(12, 20))

# 1) Training reward curves
for res in all_results:
    axes[0].plot(res["train_mean"].index, res["train_mean"], label=f"{res['name']}")
    #axes[0].fill_between(res["train_mean"].index,
    #                     res["train_mean"] - res["train_std"],
    #                     res["train_mean"] + res["train_std"],
    #                     alpha=0.2)
axes[0].set_title("Training reward (mean ± std)")
axes[0].set_xlabel("Episode")
axes[0].set_ylabel("Reward")
axes[0].legend(fontsize="small")

# 2) Evaluation reward curves
for res in all_results:
    axes[1].plot(res["eval_mean"].index, res["eval_mean"], label=f"{res['name']}")
    #axes[1].fill_between(res["eval_mean"].index,
    #                     res["eval_mean"] - res["eval_std"],
    #                     res["eval_mean"] + res["eval_std"],
    #                     alpha=0.2)
axes[1].set_title("Evaluation reward (mean ± std)")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Reward")
axes[1].legend(fontsize="small")

# 3) Summary bar plots for target proportion & time
exp_names = [res["name"] for res in all_results]
target_props = [res["target_prop"] for res in all_results]
time_means = [res["time_mean"] for res in all_results]
time_stds = [res["time_std"] for res in all_results]

# Plot proportion of runs that hit reward target
ax2 = axes[2].twinx()
axes[2].bar(exp_names, target_props, alpha=0.6, color="skyblue", label="Proportion hitting target")
axes[2].set_ylabel("Proportion hitting target")
axes[2].set_ylim(0, 1.05)

# Plot average total time
ax2.errorbar(exp_names, time_means, yerr=time_stds, fmt="o", color="red", label="Total time (mean ± std)")
ax2.set_ylabel("Total training time (s)")
axes[2].set_xticklabels(exp_names, rotation=45, ha="right")

axes[2].legend(loc="upper left")
ax2.legend(loc="upper right")



# 4) Time & Episodes to Target with two y-axes
time_means = [res["time_mean"]/60 for res in all_results]
time_stds = [res["time_std"]/60 for res in all_results]
ep_means = [res["episodes_mean"] for res in all_results]
ep_stds = [res["episodes_std"] for res in all_results]

x = np.arange(len(exp_names))
offset = 0.15  # horizontal offset to separate red and green points

ax4 = axes[3]
color_time = "tab:blue"
color_ep = "tab:orange"

# Primary axis for time (red)
ax4.errorbar(
    x - offset, time_means, yerr=time_stds,
    fmt="o", color=color_time, capsize=5, label="Total time (mean ± std)"
)
ax4.set_ylabel("Total time (min)", color=color_time)
ax4.tick_params(axis="y", labelcolor=color_time)

# Secondary axis for episodes (green)
ax4b = ax4.twinx()
ax4b.errorbar(
    x + offset, ep_means, yerr=ep_stds,
    fmt="s", color=color_ep, capsize=5, label="Episodes to target (mean ± std)"
)
ax4b.set_ylabel("Episodes to target (scaled to n_envs=4)", color=color_ep)
ax4b.tick_params(axis="y", labelcolor=color_ep)

# X-axis labels
ax4.set_xticks(x)
ax4.set_xticklabels(exp_names, rotation=45, ha="right")
ax4.set_title("Total time and episodes to target")

# Add legends for both axes
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1 + lines2, labels1 + labels2, loc="upper center", fontsize="small")





plt.tight_layout()
plt.show()


In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ----- Load and preprocess (same as your current code) -----
base_log_dir = "log_data/sac_parallel_evaluation_gs_0500"
n_repeats = 5
number_of_envs = 4

train_dfs = []
eval_dfs = []

for run_idx in range(n_repeats):
    log_dir = os.path.join(base_log_dir, f"run_{run_idx}")
    train_df = pd.read_csv(os.path.join(log_dir, "train_rewards.csv"))
    train_df['run'] = run_idx
    train_dfs.append(train_df)

    eval_df = pd.read_csv(os.path.join(log_dir, "eval_rewards.csv"))
    eval_df['run'] = run_idx
    eval_dfs.append(eval_df)

train_all = pd.concat(train_dfs, ignore_index=True)
eval_all = pd.concat(eval_dfs, ignore_index=True)

train_all['timestamp'] = pd.to_datetime(train_all['timestamp'], format='%H:%M:%S')
eval_all['timestamp'] = pd.to_datetime(eval_all['timestamp'], format='%H:%M:%S')

def compute_elapsed(df):
    elapsed_list = []
    for run in df['run'].unique():
        subset = df[df['run'] == run]
        start = subset['timestamp'].min()
        elapsed = (subset['timestamp'] - start).dt.total_seconds()
        elapsed_list.append(elapsed)
    return pd.concat(elapsed_list).sort_index()

train_all['elapsed_seconds'] = compute_elapsed(train_all)
eval_all['elapsed_seconds'] = compute_elapsed(eval_all)

# ----- Aggregate data -----
train_grouped = train_all.groupby(['run', 'episode']).agg({
    'reward': 'mean',
    'elapsed_seconds': 'mean'
}).reset_index()

eval_grouped = eval_all.groupby(['run', 'episode']).agg({
    'reward': 'mean',
    'elapsed_seconds': 'mean'
}).reset_index()

# Compute mean and std across runs for each episode
train_summary = train_grouped.groupby('episode').agg({
    'reward': ['mean', 'std'],
    'elapsed_seconds': 'mean'
}).reset_index()
train_summary.columns = ['episode', 'reward_mean', 'reward_std', 'elapsed_seconds']

# ----- Plotting -----
fig, ax1 = plt.subplots(figsize=(14, 10))

# Plot all runs in light grey
for run in train_grouped['run'].unique():
    run_data = train_grouped[train_grouped['run'] == run]
    ax1.plot(run_data['episode'], run_data['reward'], color='grey', alpha=0.5)

# Plot mean training reward with std fill
ax1.plot(train_summary['episode'], train_summary['reward_mean'], color='blue', label='Avg train reward')
#ax1.fill_between(train_summary['episode'],
#                 train_summary['reward_mean'] - train_summary['reward_std'],
#                 train_summary['reward_mean'] + train_summary['reward_std'],
#                 color='blue', alpha=0.2)

# Plot evaluation rewards as blue dots
for run in eval_grouped['run'].unique():
    run_data = eval_grouped[eval_grouped['run'] == run]
    ax1.plot(run_data['episode'], run_data['reward'], linestyle='solid',
             color='green', alpha=0.8, label=None)

# Label only once for evaluation reward
ax1.plot([], [], marker='o', linestyle='None', color='blue', label='Evaluation reward')

# Legend
ax1.plot([], [], color='grey', alpha=0.5, label='Training runs')
ax1.legend()

# Axis labels
ax1.set_ylabel('Reward')
ax1.set_xlabel('Episode')

# ----- Add secondary x-axes -----
# Simulated time = 10 * episode
def episode_to_simtime(x):
    return x * 10
def simtime_to_episode(x):
    return x / 10

secax_sim = ax1.secondary_xaxis(-0.1, functions=(episode_to_simtime, simtime_to_episode))
secax_sim.set_xlabel('Simulated time (s)')

# Compute time (elapsed_seconds) from average
def episode_to_elapsed(x):
    return np.interp(x, train_summary['episode'], train_summary['elapsed_seconds'])
def elapsed_to_episode(x):
    return np.interp(x, train_summary['elapsed_seconds'], train_summary['episode'])

secax_time = ax1.secondary_xaxis(-0.2, functions=(episode_to_elapsed, elapsed_to_episode))
secax_time.set_xlabel('Compute time (s)')

plt.title("Training and Evaluation Rewards (All Runs)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the CSV
base_log_dir = "log_data/sac_parallel_evaluation"

df = pd.read_csv(os.path.join(base_log_dir,'runtime_log.csv'))

# Set up bar positions and labels
x = range(len(df))
bar_labels = [f"Run {i}" for i in x]  # You can replace with your own labels

# Plot
fig, ax = plt.subplots()

bottom = [0] * len(df)
#colors = ['skyblue', 'orange', 'green']  # Optional: set your own colors

for i, column in enumerate(df.columns):
    ax.bar(x, df[column], bottom=bottom, label=column)
    bottom = [bottom[j] + df[column][j] for j in range(len(df))]

# Customize plot
ax.set_xticks(x)
ax.set_xticklabels(bar_labels)
ax.set_ylabel("Time (s)")
ax.set_title("Time Breakdown per Run Across Training Stages")
ax.legend()

plt.tight_layout()
plt.show()


# Simulate swing using SAC controller

In [ ]:
#Changed from TkAgg which does not work on macOS with homebrew
#matplotlib.use('Agg')
# initialize the pendulum
pendulum = PendulumPlant(mass=mass,
                         length=length,
                         damping=damping,
                         gravity=gravity,
                         coulomb_fric=coulomb_fric,
                         inertia=inertia,
                         torque_limit=torque_limit)

sim = Simulator(plant=pendulum)

# get the controller we trained earlier
#model_path = "data_models/sac_model.zip"
#model_path = "log_data/sac_training/best_model/best_model.zip"
#model_path = "log_data/sac_parallel_training/best_model/best_model.zip"
model_path = "log_data/sac_parallel_evaluation/run_0/best_model.zip"
#model_path = "best_model/best_model.zip"
controller = SacController(model_path=model_path,
                           torque_limit=torque_limit,
                           use_symmetry=False,
                           state_representation=3,
                           deterministic=True)

# simulate
x0_sim = [0.0, 0.0]
dt = 0.02
t_final = 10
integrator = "runge_kutta"

T, X, U = sim.simulate(t0=0.0,
                                   x0=x0_sim,
                                   tf=t_final,
                                   dt=dt,
                                   controller=controller,
                                   integrator=integrator)


fig, ax = plt.subplots(3, 1, figsize=(18, 6), sharex="all")
print(f"Final torque: {U[-1]}")
ax[0].plot(T, np.asarray(X).T[0], label="theta")
ax[0].hlines(y=[-np.pi, np.pi], xmin=0, xmax = t_final, linestyles = ':', colors='g')
ax[0].set_ylabel("angle [rad]")
ax[0].legend(loc="best")
ax[1].plot(T, np.asarray(X).T[1], label="theta dot")
ax[1].set_ylabel("angular velocity [rad/s]")
ax[1].legend(loc="best")
ax[2].plot(T, np.asarray(U).flatten(), label="u")
ax[2].set_xlabel("time [s]")
ax[2].set_ylabel("input torque [Nm]")
ax[2].hlines(y=0, xmin=0, xmax=t_final, linestyles = ':', colors='g')
ax[2].legend(loc="best")
plt.show()


# SAC test of current policy in gym environment

In [ ]:
from stable_baselines3 import SAC
import numpy as np
import matplotlib.pyplot as plt

from pendulum_plant.pendulum_plant import PendulumPlant
from pendulum_plant.simulation import Simulator
from pendulum_plant.gym_environment import SimplePendulumEnv

# === Load trained model ===
#model_path = "log_data/sac_parallel_training/best_model/best_model.zip"
model = SAC.load(model_path)

# === Define environment (must match training) ===
mass = 0.06
length = 0.1
damping = 0.0004
torque_limit = 0.02
coulomb_fric = 0.0
gravity = 9.81
inertia = mass * length**2

dt = 0.02
integrator = "runge_kutta"
max_steps = 500
reward_type = "combined_reward"
state_representation = 3
random_init = "False"

pendulum = PendulumPlant(mass=mass, length=length, damping=damping,
                         gravity=gravity, coulomb_fric=coulomb_fric,
                         inertia=inertia, torque_limit=torque_limit)
simulator = Simulator(plant=pendulum)
env = SimplePendulumEnv(simulator=simulator,
                        max_steps=max_steps,
                        reward_type=reward_type,
                        state_target_epsilon = [0.05, 0.05],
                        dt=dt,
                        integrator=integrator,
                        state_representation=state_representation,
                        validation_limit=40_000,
                        scale_action=True,  # must match training
                        random_init=random_init)

# === Run rollout ===
obs = env.reset()
obs_list, action_list, reward_list = [], [], []

for i in range(max_steps):
    print(obs)
    action, _ = model.predict(obs, deterministic=True)
    #if i > 200 and i<210:
    #    action = np.array([-1])
    #if i > 400 and i<410:
    #    action = np.array([1])
    obs, reward, done, _ = env.step(action)
    
    obs_list.append(obs)
    action_list.append(action)
    reward_list.append(reward)
    
    if done:
        break
print("Total Reward: ", sum(reward_list))
# === Plot results ===
obs_array = np.array(obs_list)
action_array = np.array(action_list).flatten()
time = np.arange(len(obs_array)) * dt

fig, ax = plt.subplots(3, 1, figsize=(12, 6), sharex=True)
ax[0].plot(time, obs_array[:, -1], label="Angular velocity")
ax[0].set_ylabel("θ̇ [rad/s]")
ax[0].legend()

if state_representation == 3:
    theta = np.arctan2(obs_array[:,1], obs_array[:,0])
    ax[1].plot(time, theta, label="Angle (reconstructed)")
    ax[1].hlines([-np.pi, np.pi], 0, time[-1], colors='gray', linestyles='dashed')
    ax[1].set_ylabel("θ [rad]")
    ax[1].legend()
else:
    ax[1].plot(time, obs_array[:,0], label="Angle")
    ax[1].set_ylabel("θ [rad]")
    ax[1].legend()

ax[2].plot(time, action_array, label="Torque")
ax[2].set_ylabel("τ [Nm]")
ax[2].set_xlabel("Time [s]")
ax[2].hlines(0, 0, time[-1], colors='gray', linestyles='dashed')
ax[2].legend()

plt.tight_layout()
plt.show()


# SAC test of both deterministic and non-deterministic policy

In [ ]:


# --- Setup simulation parameters ---
x0_sim = [0.0, 0.0]
dt = 0.02
t_final = 20
integrator = "runge_kutta"
n_stochastic_trials = 4  # Number of stochastic rollouts

# --- Run deterministic rollout ---
controller.deterministic = True
T_det, X_det, U_det = sim.simulate(t0=0.0, x0=x0_sim, tf=t_final, dt=dt,
                                   controller=controller, integrator=integrator)

# --- Run stochastic rollouts ---
stochastic_rollouts = []
controller.deterministic = False
for _ in range(n_stochastic_trials):
    T, X, U = sim.simulate(t0=0.0, x0=x0_sim, tf=t_final, dt=dt,
                           controller=controller, integrator=integrator)
    stochastic_rollouts.append((T, X, U))

# --- Get alpha value if available ---
#try:
#    alpha = float(controller.model.ent_coef)
#except:
#    alpha = "unknown"

# --- Plotting ---
fig, ax = plt.subplots(3, 1, figsize=(18, 10), sharex="all")
#ax[0].set_title(f"Deterministic (top) and {n_stochastic_trials} Stochastic Rollouts (α = {alpha:.3f})")

# Plot deterministic rollout
X_det = np.array(X_det)
U_det = np.array(U_det)
ax[0].plot(T_det, X_det[:, 0], color='black', label="theta (deterministic)", linewidth=2)
ax[0].hlines(y=[-np.pi, np.pi], xmin=0, xmax=t_final, linestyles=':', colors='g')
ax[0].set_ylabel("angle [rad]")
ax[0].legend(loc="upper right")

# Plot stochastic rollouts
colors = plt.cm.viridis(np.linspace(0.3, 1.0, n_stochastic_trials))
for i, (T, X, U) in enumerate(stochastic_rollouts):
    X = np.array(X)
    U = np.array(U)
    ax[1].plot(T, X[:, 0], label=f"θ trial {i+1}", color=colors[i])
    ax[2].plot(T, U.flatten(), label=f"u trial {i+1}", color=colors[i])

ax[1].hlines(y=[-np.pi, np.pi], xmin=0, xmax=t_final, linestyles=':', colors='g')
ax[1].set_ylabel("angle [rad] (stochastic)")
ax[1].legend(loc="best")

ax[2].hlines(y=0, xmin=0, xmax=t_final, linestyles=':', colors='g')
ax[2].set_ylabel("torque [Nm]")
ax[2].set_xlabel("time [s]")
ax[2].legend(loc="best")

plt.tight_layout()
plt.show()


# Policy Visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def visualize_policy_with_stabilization_view(controller, 
                                             theta_range=(-np.pi, np.pi), 
                                             vel_range=(-8, 8), 
                                             stab_range=0.6,
                                             bins=200,
                                             zoom_bins=100):
    """
    Plots two heatmaps:
    1. Full (theta, velocity) policy
    2. Zoomed view around stabilization point ±π, shown as deviation from upright

    Parameters
    ----------
    controller : ddpg_controller
        The DDPG controller.
    theta_range : tuple
        Range for theta in full plot.
    vel_range : tuple
        Range for velocity in both plots.
    stab_range : float
        Maximum deviation (in radians) from the upright position (π or -π).
    bins : int
        Resolution for full plot.
    zoom_bins : int
        Resolution for zoom plot.
    """
    def compute_actions(theta_vals, vel_vals):
        actions = np.zeros((len(vel_vals), len(theta_vals)))
        for i, th in enumerate(theta_vals):
            for j, vel in enumerate(vel_vals):
                action = controller.get_control_output(meas_pos=th, meas_vel=vel)
                actions[j, i] = action
        return actions

    # === Full policy ===
    thetas = np.linspace(*theta_range, bins)
    vels = np.linspace(*vel_range, bins)
    full_actions = compute_actions(thetas, vels)

    # === Stabilization view around ±π ===
    delta_theta_vals = np.linspace(-stab_range, stab_range, zoom_bins)
    vels_zoom = np.linspace(-4, 4, zoom_bins)

    # Map delta_theta to wrapped angles near ±π
    theta_left = -np.pi + delta_theta_vals  # around -π
    theta_right = np.pi - delta_theta_vals  # around +π

    # Merge both into one batch
    theta_zoom = np.concatenate([theta_left, theta_right])
    delta_zoom = np.concatenate([delta_theta_vals, delta_theta_vals])  # for x-axis
    vels_zoom_full = np.tile(vels_zoom, 2)

    actions_zoom = np.zeros((zoom_bins, 2 * zoom_bins))
    for i, dt in enumerate(delta_theta_vals):
        for j, vel in enumerate(vels_zoom):
            a_left = controller.get_control_output(meas_pos=-np.pi + dt, meas_vel=vel)
            a_right = controller.get_control_output(meas_pos=np.pi + dt, meas_vel=vel)
            actions_zoom[j, i] = a_left
            actions_zoom[j, i + zoom_bins] = a_right

    # === Plotting ===
    fig, axs = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

    # Full plot
    im1 = axs[0].imshow(full_actions, extent=[*theta_range, *vel_range],
                        origin='lower', aspect='auto', cmap='coolwarm')
    axs[0].set_title("Full Policy")
    axs[0].set_xlabel("Theta (rad)")
    axs[0].set_ylabel("Angular Velocity")
    fig.colorbar(im1, ax=axs[0], label='Torque')

    # Stabilization plot
    x_extent = [-stab_range, stab_range]
    im2 = axs[1].imshow(actions_zoom, extent=[x_extent[0], x_extent[1], -4, 4],
                        origin='lower', aspect='auto', cmap='coolwarm')
    axs[1].set_title("Stabilization Near ±π")
    axs[1].set_xlabel("ΔTheta from ±π (rad)")
    fig.colorbar(im2, ax=axs[1], label='Torque')

    plt.suptitle("DDPG Policy: Full and Stabilization Region Views")
    plt.tight_layout()
    plt.show()
visualize_policy_with_stabilization_view(controller)

In [ ]:
import matplotlib.pyplot as plt

# Data
labels = ['Hardware Run']
experience_collecting = [655.814772605896/60]
policy_updates = [589.4724380970001/60]
evaluation = [113.8405487537384/60]

# Create stacked bar chart
plt.bar(labels, experience_collecting, label='Experience collecting')
plt.bar(labels, policy_updates, bottom=experience_collecting, label='Policy Updates')
plt.bar(labels, evaluation, 
        bottom=[i+j for i,j in zip(experience_collecting, policy_updates)],
        label='Evaluation')

# Labels and title
plt.ylabel('Time (minutes)')
plt.title('Training Time Breakdown')
plt.legend()

plt.show()
